# Prepare Source 4: Bradley data from Austermeier et al. (2025)

This notebook creates the single analysis-ready Source 4 input used by the combined melting-point workflow:

`input_datasets/4_Bradley_Austermeier_2025.csv`

It uses:

- all ten files in `original_curated/train_without_data_augmentation/`;
- `original_curated/test_predictions/consensus_without_data_augmentation.csv`.

Only experimental melting-point observations are retained. Prediction columns are ignored. The inherited train/test labels describe the source study and are not used as this project's final model split.

Deduplication is performed on the exact pair `SMILES` + `MP`. Repeated copies of the same observation are removed, while different reported MPs for the same SMILES are deliberately preserved for later consensus handling in `combine_data_process.ipynb`.

This notebook does not canonicalize SMILES. Canonicalization and cross-source consensus are performed in the main combination notebook.


In [1]:
from pathlib import Path
import hashlib
import re

import pandas as pd
from IPython.display import display

data_dir_candidates = [Path.cwd(), Path.cwd() / '0_data', Path.cwd().parent / '0_data']
DATA_DIR = next(
    path.resolve()
    for path in data_dir_candidates
    if (path / 'original_curated').exists()
)

TRAIN_DIR = DATA_DIR / 'original_curated' / 'train_without_data_augmentation'
TEST_FILE = (
    DATA_DIR / 'original_curated' / 'test_predictions'
    / 'consensus_without_data_augmentation.csv'
)
VALIDATION_DIR = DATA_DIR / 'original_curated' / 'val'
OUTPUT_DIR = DATA_DIR / 'input_datasets'
OUTPUT_FILE = OUTPUT_DIR / '4_Bradley_Austermeier_2025.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data directory: {DATA_DIR}')
print(f'Training directory: {TRAIN_DIR}')
print(f'Test file: {TEST_FILE}')
print(f'Output file: {OUTPUT_FILE}')


Data directory: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/0_data
Training directory: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/0_data/original_curated/train_without_data_augmentation
Test file: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/0_data/original_curated/test_predictions/consensus_without_data_augmentation.csv
Output file: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/0_data/input_datasets/4_Bradley_Austermeier_2025.csv


## Load the non-augmented training folds

The fold number is extracted from each filename so the input order is deterministic: `train1_curated.csv` through `train10_curated.csv`.


In [2]:
def training_fold_number(path):
    match = re.fullmatch(r'train(\d+)_curated\.csv', path.name)
    if match is None:
        raise ValueError(f'Unexpected training filename: {path.name}')
    return int(match.group(1))


train_files = sorted(TRAIN_DIR.glob('train*_curated.csv'), key=training_fold_number)
if not train_files:
    raise FileNotFoundError(f'No training CSV files found in {TRAIN_DIR}')

fold_numbers = [training_fold_number(path) for path in train_files]
if fold_numbers != list(range(1, 11)):
    raise ValueError(f'Expected training folds 1–10, found: {fold_numbers}')

training_frames = []
training_audit_rows = []

for path in train_files:
    frame = pd.read_csv(path, usecols=['SMILES', 'MP'])
    frame['MP'] = pd.to_numeric(frame['MP'], errors='raise')
    frame['Original_subset'] = 'train'
    frame['Original_file'] = path.name
    training_frames.append(frame)
    training_audit_rows.append({
        'Fold': training_fold_number(path),
        'Original_file': path.name,
        'Rows': len(frame),
        'Unique_raw_SMILES': frame['SMILES'].nunique(dropna=False),
    })

training_observations = pd.concat(training_frames, ignore_index=True)
training_audit = pd.DataFrame(training_audit_rows)

display(training_audit)
print(f'Training rows across folds: {len(training_observations):,}')
print(f'Unique raw training SMILES: {training_observations["SMILES"].nunique(dropna=False):,}')


,Fold,Original_file,Rows,Unique_raw_SMILES
0,1,train1_curated.csv,13734,13715
1,2,train2_curated.csv,13734,13721
2,3,train3_curated.csv,13734,13716
3,4,train4_curated.csv,13734,13717
4,5,train5_curated.csv,13734,13717
5,6,train6_curated.csv,13734,13715
6,7,train7_curated.csv,13734,13716
7,8,train8_curated.csv,13734,13719
8,9,train9_curated.csv,13734,13720
9,10,train10_curated.csv,13734,13717


Training rows across folds: 137,340
Unique raw training SMILES: 17,633


## Load the study test observations

The test file contains model predictions, but only `SMILES` and the experimental `exp MP` field are used. The experimental field is renamed to `MP` for a common Source 4 schema.


In [3]:
if not TEST_FILE.exists():
    raise FileNotFoundError(f'Missing requested test file: {TEST_FILE}')

test_observations = pd.read_csv(TEST_FILE, usecols=['SMILES', 'exp MP'])
test_observations = test_observations.rename(columns={'exp MP': 'MP'})
test_observations['MP'] = pd.to_numeric(test_observations['MP'], errors='raise')
test_observations['Original_subset'] = 'test'
test_observations['Original_file'] = TEST_FILE.name

print(f'Test rows: {len(test_observations):,}')
print(f'Unique raw test SMILES: {test_observations["SMILES"].nunique(dropna=False):,}')


Test rows: 1,961
Unique raw test SMILES: 1,961


## Combine and remove only exact duplicate observations

A SMILES can retain more than one row when its numeric MP values differ. The conflict summary below is an audit of those retained cases; it does not resolve them.


In [4]:
all_observations = pd.concat(
    [training_observations, test_observations],
    ignore_index=True,
)

missing_smiles = int(all_observations['SMILES'].isna().sum())
missing_mp = int(all_observations['MP'].isna().sum())
if missing_smiles or missing_mp:
    raise ValueError(
        f'Missing required values: SMILES={missing_smiles:,}, MP={missing_mp:,}'
    )

exact_duplicate_mask = all_observations.duplicated(
    subset=['SMILES', 'MP'], keep='first'
)
exact_duplicates_removed = int(exact_duplicate_mask.sum())

bradley_data = (
    all_observations.loc[~exact_duplicate_mask, ['SMILES', 'MP']]
    .sort_values(['SMILES', 'MP'], kind='stable')
    .reset_index(drop=True)
)

mp_counts = bradley_data.groupby('SMILES')['MP'].nunique()
conflicting_smiles = mp_counts.loc[mp_counts > 1].index
conflict_summary = (
    bradley_data.loc[bradley_data['SMILES'].isin(conflicting_smiles)]
    .groupby('SMILES', as_index=False)
    .agg(
        N_distinct_MP=('MP', 'nunique'),
        MP_values=('MP', lambda values: sorted(set(map(float, values)))),
        MP_min=('MP', 'min'),
        MP_max=('MP', 'max'),
    )
    .sort_values(['N_distinct_MP', 'SMILES'], ascending=[False, True], kind='stable')
    .reset_index(drop=True)
)

train_smiles = set(training_observations['SMILES'])
test_smiles = set(test_observations['SMILES'])
overlap_smiles = train_smiles & test_smiles
overlap_pairs = (
    training_observations[['SMILES', 'MP']].drop_duplicates()
    .merge(
        test_observations[['SMILES', 'MP']].drop_duplicates(),
        on=['SMILES', 'MP'],
        how='inner',
    )
)

summary = pd.DataFrame({
    'Metric': [
        'Input training rows',
        'Input test rows',
        'Combined input rows',
        'Exact duplicate SMILES–MP rows removed',
        'Output SMILES–MP observations',
        'Unique raw SMILES in output',
        'SMILES with multiple distinct MPs retained',
        'Raw SMILES shared by train and test',
        'Exact SMILES–MP pairs shared by train and test',
    ],
    'Count': [
        len(training_observations),
        len(test_observations),
        len(all_observations),
        exact_duplicates_removed,
        len(bradley_data),
        bradley_data['SMILES'].nunique(),
        len(conflict_summary),
        len(overlap_smiles),
        len(overlap_pairs),
    ],
})

display(summary)
display(conflict_summary)


,Metric,Count
0,Input training rows,137340
1,Input test rows,1961
2,Combined input rows,139301
3,Exact duplicate SMILES–MP rows removed,119692
4,Output SMILES–MP observations,19609
5,Unique raw SMILES in output,19588
6,SMILES with multiple distinct MPs retained,21
7,Raw SMILES shared by train and test,6
8,Exact SMILES–MP pairs shared by train and test,2


,SMILES,N_distinct_MP,MP_values,MP_min,MP_max
0,CC(=O)OC[C@H]1O[C@H](OC(=O)C)[C@@H]([C@H]([C@@...,2,"[111.0, 113.3]",111.0,113.3
1,CC(CCC[C@H]([C@H]1CC[C@@H]2[C@]1(C)CC[C@H]1[C@...,2,"[148.5, 149.0]",148.5,149.0
2,CO[C@H]1O[C@H](CO)[C@H]([C@@H]([C@H]1O)O)O,2,"[168.0, 169.0]",168.0,169.0
3,Nc1ccc(cc1)N=Nc1ccccc1,2,"[124.0, 127.0]",124.0,127.0
4,O=C(Nc1c(C)cccc1C)CN1CCN(CC1)CCCC(c1ccc(cc1)F)...,2,"[159.0, 160.0]",159.0,160.0
5,O=C1CC[C@H]2C(=C1)CC[C@@H]1[C@@H]2CC[C@]2([C@H...,2,"[112.0, 118.0]",112.0,118.0
6,O=C1CC[C@]2(C(=C1)CC[C@@H]1[C@@H]2CC[C@]2([C@H...,2,"[153.0, 155.0]",153.0,155.0
7,O=C1CC[C@]2(C(=C1)[C@@H](C)C[C@@H]1[C@@H]2CC[C...,2,"[206.5, 208.0]",206.5,208.0
8,OCC(=O)[C@@]1(O)CC[C@@H]2[C@]1(C)C[C@H](O)[C@H...,2,"[214.0, 223.0]",214.0,223.0
9,OCC(=O)[C@@]1(O)[C@@H](C)C[C@@H]2[C@]1(C)C[C@H...,2,"[232.0, 232.5]",232.0,232.5


## Confirm that validation files add no new structures

The ten training-fold files collectively cover the source study's cross-validation pool. This check confirms that the separately stored validation folds do not introduce raw SMILES absent from the training-fold union. Validation rows are therefore not appended again.


In [5]:
validation_files = sorted(
    VALIDATION_DIR.glob('val*_curated.csv'),
    key=lambda path: int(re.fullmatch(r'val(\d+)_curated\.csv', path.name).group(1)),
)

if validation_files:
    validation_smiles = set()
    for path in validation_files:
        validation_smiles.update(
            pd.read_csv(path, usecols=['SMILES'])['SMILES'].dropna()
        )
    validation_only_smiles = validation_smiles - train_smiles
    print(f'Unique validation SMILES: {len(validation_smiles):,}')
    print(f'Validation SMILES absent from training-fold union: {len(validation_only_smiles):,}')
    if validation_only_smiles:
        raise ValueError(
            'Validation files contain structures absent from the training-fold union. '
            'Review the intended Source 4 coverage before writing the output.'
        )
else:
    print('No validation files found; validation-coverage check skipped.')


Unique validation SMILES: 16,193
Validation SMILES absent from training-fold union: 0


## Write and validate the Source 4 snapshot

The output intentionally contains only `SMILES` and `MP`. Original paths, filenames, and the preparation method should be recorded in `input_datasets/source_manifest.csv`.


In [6]:
bradley_data.to_csv(OUTPUT_FILE, index=False)

saved = pd.read_csv(OUTPUT_FILE)
saved['MP'] = pd.to_numeric(saved['MP'], errors='raise')

assert list(saved.columns) == ['SMILES', 'MP']
assert saved['SMILES'].notna().all()
assert saved['MP'].notna().all()
assert not saved.duplicated(['SMILES', 'MP']).any()
assert saved.equals(bradley_data)

saved_mp_counts = saved.groupby('SMILES')['MP'].nunique()
assert int(saved_mp_counts.gt(1).sum()) == len(conflict_summary)


def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


input_inventory = pd.DataFrame([
    {
        'Role': 'train fold',
        'Original_path': str(path.relative_to(DATA_DIR)),
        'Rows': len(frame),
        'SHA256': sha256(path),
    }
    for path, frame in zip(train_files, training_frames)
] + [{
    'Role': 'test observations',
    'Original_path': str(TEST_FILE.relative_to(DATA_DIR)),
    'Rows': len(test_observations),
    'SHA256': sha256(TEST_FILE),
}])

display(input_inventory)
print(f'Wrote {len(saved):,} observations to: {OUTPUT_FILE}')
print(f'Unique raw SMILES: {saved["SMILES"].nunique():,}')
print(f'SMILES retaining multiple MP values: {saved_mp_counts.gt(1).sum():,}')
print(f'Output SHA256: {sha256(OUTPUT_FILE)}')
print('All Source 4 validation checks passed.')


,Role,Original_path,Rows,SHA256
0,train fold,original_curated/train_without_data_augmentati...,13734,4adc1acc2f5137810e583154bd530697a811ee6f20efd5...
1,train fold,original_curated/train_without_data_augmentati...,13734,cf67296b1c227aeffcd9ff15ee22741754a5a2495b6b91...
2,train fold,original_curated/train_without_data_augmentati...,13734,aa3f89409e8b084c7b42635156e6de5858bea32755b9a1...
3,train fold,original_curated/train_without_data_augmentati...,13734,2a5f0fafe8d40af51e5e463d9a591737942523547579f0...
4,train fold,original_curated/train_without_data_augmentati...,13734,681ac8d9634e5099ea9c6d814399b371051abe938f6f00...
5,train fold,original_curated/train_without_data_augmentati...,13734,b53de2b7f08d6eeed150214dfcdd4a132dc78d726dd89e...
6,train fold,original_curated/train_without_data_augmentati...,13734,b42ed52d157b58d832a514af8aa85b606f0470d0c1d871...
7,train fold,original_curated/train_without_data_augmentati...,13734,476f5f6f079132ba59efd65567c0afe14274817b1468b2...
8,train fold,original_curated/train_without_data_augmentati...,13734,50ae5c48a705ee73b5e41495cf4049b93003bc4f9b3116...
9,train fold,original_curated/train_without_data_augmentati...,13734,daa3faee4774d28e4256135b1ac1287dc9a4d99769eb9d...


Wrote 19,609 observations to: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/0_data/input_datasets/4_Bradley_Austermeier_2025.csv
Unique raw SMILES: 19,588
SMILES retaining multiple MP values: 21
Output SHA256: ecf06847acdb43c3aaec73b6d743d34862f180fd63f959484eebe77b9a6ac78e
All Source 4 validation checks passed.


## Next step

After this notebook succeeds, update `combine_data_process.ipynb` so Source 4 is loaded from:

`input_datasets/4_Bradley_Austermeier_2025.csv`

using the `MP` column. Then rerun the complete combination workflow because preserving conflicting Source 4 observations and collapsing exact train/test duplicates can change consensus results and final dataset counts.
